## Dataloader

In [4]:
import os
import numpy as np
import jax
import jax.numpy as jnp
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

from jepax.model.masker import IJEPAMasker  # your file


def numpy_collate(batch):
    return np.stack([np.array(x) for x, _ in batch]), np.array([y for _, y in batch])


def get_cifar10_loader(data_dir, batch_size=64, train=True):
    dataset = datasets.CIFAR10(
        data_dir, train=train, download=True,
        transform=transforms.ToTensor()
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=train,
        drop_last=train,
        collate_fn=numpy_collate,
    )


if __name__ == '__main__':
    
    loader = get_cifar10_loader('~/data', batch_size=64)
    masker = IJEPAMasker(height=32, width=32, patch_size=4)
    
    key = jax.random.PRNGKey(0)
    M = 4  # number of prediction masks
    
    for images, labels in loader:
        key, subkey = jax.random.split(key)
        keys = jax.random.split(subkey, images.shape[0])
        
        ctx_masks, pred_masks = jax.vmap(lambda k: masker(k, M, flatten=True))(keys)
        
        print('images:', images.shape)           # (64, 3, 32, 32)
        print('ctx_masks:', ctx_masks.shape)     # (64, 64)
        print('pred_masks:', pred_masks.shape)   # (64, 4, 64)
        break

/var/folders/82/kqk386yx56s3vbqsdcrdr_wc0000gn/T/ipykernel_41942/4172804979.py:12: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  return np.stack([np.array(x) for x, _ in batch]), np.array([y for _, y in batch])


VmapTracer<float32[]>
images: (64, 3, 32, 32)
ctx_masks: (64, 64)
pred_masks: (64, 4, 64)
